# Side-Chain Rotamers and Packing - Student Notebook

Given a peptide backbone, where do the **side chains** go? This notebook lets you **play with** the answer:

1. side chains are described by **chi torsion angles**, which cluster into a small number of **rotamers**;
2. picking one rotamer per residue is a **discrete combinatorial optimization** problem;
3. we solve it three ways - greedy, **Dead-End Elimination**, and **simulated annealing** - and check them against the provable global optimum by brute force.

Companion reading: `Rotamer_tutorial.md` (or the Chinese version).

**Prerequisites:** `numpy`, `matplotlib`, `rdkit`.

Run the cells top to bottom, then try the exercises at the end.

## 0. Setup

Run this notebook from the `rotamer/` directory so that the `core` package imports.

In [ ]:
import os, sys, itertools

# Make sure the directory containing `core/` is importable.
if not os.path.isdir('core'):
    if os.path.isdir('rotamer/core'):
        os.chdir('rotamer')                  # started from the repo root
    else:
        raise RuntimeError('Run this notebook from the rotamer/ directory.')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt

from core import (
    CHI_DEFINITIONS, CHI_STATES, n_chi, chi_atom_names,
    Rotamer, enumerate_rotamers,
    Peptide, mmff_energy, minimize,
    score_residue_rotamers, build_low_energy_conformation,
    build_energy_matrix, dead_end_elimination, simulated_annealing, solve_rotamers,
)
print('imports OK, cwd =', os.getcwd())

## 1. A rotamer is a set of chi angles

Each side chain rotates about its own single bonds. Chi-1 is the `N-CA-CB-*` torsion, chi-2 the next one out, and so on. To *set* a chi angle you need the four atoms that define it - that is exactly what `CHI_DEFINITIONS` stores.

Note ALA and GLY have no rotatable side chain, and PRO's chi angles sit inside a five-membered ring, so all three are treated as rigid.

In [ ]:
for resname in ('VAL', 'PHE', 'LEU', 'LYS', 'ARG', 'ALA', 'PRO'):
    quartets = chi_atom_names(resname)
    if not quartets:
        print(f'{resname}: rigid (no rotatable chi)')
        continue
    print(f'{resname}: n_chi = {n_chi(resname)}')
    for k, q in enumerate(quartets, 1):
        print(f'    chi{k}: ' + '-'.join(q))

## 2. The staggered rotamer library

Chi angles are not uniformly distributed - they cluster near the three **staggered** positions around a tetrahedral bond:

| state | name | angle |
|-------|------|-------|
| `p` | gauche+ | +60 deg |
| `t` | trans | 180 deg |
| `m` | gauche- | -60 deg |

So a rotamer gets a short name like `"mt"` = (chi1 = -60, chi2 = 180). The library size grows as `3**n_chi`, which is why we only *vary* the first `max_chi` angles (chi1 and chi2 dominate side-chain identity) and leave deeper ones trans.

In [ ]:
print('CHI_STATES:', CHI_STATES)

print('\nLYS rotamers with max_chi=2 (LYS has 4 chi angles):')
for rot in enumerate_rotamers('LYS', max_chi=2):
    chi_txt = ', '.join(f'{c:+6.1f}' for c in rot.chi)
    print(f'  {rot.name}:  chi = ({chi_txt})     <- chi3, chi4 stay trans')

print('\nlibrary size vs max_chi, for LYS:')
for m in (1, 2, 3, 4):
    print(f'  max_chi={m}: {len(enumerate_rotamers("LYS", max_chi=m)):3d} rotamers  (3**{min(m, 4)})')

## 3. Build a peptide

`Peptide.from_sequence` builds a 3D structure with RDKit (ETKDG embedding + a short MMFF relaxation). We use **KLVFF**, the hydrophobic core motif of amyloid-beta, because it has five flexible side chains including two stacking phenylalanines.

In [ ]:
sequence = 'KLVFF'
peptide = Peptide.from_sequence(sequence)

print(f'{sequence}: {peptide.mol.GetNumAtoms()} atoms (hydrogens included)')
print(f'MMFF energy of the embedded start: {mmff_energy(peptide.mol):.2f} kcal/mol\n')

print(f"{'res':>6} {'n_chi':>6} {'rotamers':>9}   current chi angles (deg)")
combos = 1
for res in peptide.residues:
    rots = enumerate_rotamers(res.name, max_chi=2)
    chi_txt = ', '.join(f'{c:+7.1f}' for c in peptide.get_all_chi(res.number)) or '-'
    print(f'{res.name}{res.number:<3d} {res.n_chi:>6} {len(rots):>9}   {chi_txt}')
    combos *= max(1, len(rots))

print(f'\nrotamer combinations to choose from: {combos:,}')
print('Small enough to brute force here - but it grows as 9**n_residues,')
print('so a real protein is hopeless without a smarter algorithm.')

## 4. Setting a rotamer, and checking it took

`set_rotamer` writes each chi angle from chi1 outward, so rotating an inner bond does not disturb an already-placed outer angle. Let's set a rotamer and read the angles straight back as a round-trip test.

In [ ]:
trial = peptide.copy()                       # never mutate the original
target = Rotamer(name='mt', chi=(-60.0, 180.0, 180.0, 180.0))
trial.set_rotamer(1, target)                 # residue 1 = LYS

readback = trial.get_all_chi(1)
print('requested:', target.chi)
print('read back:', tuple(round(c, 1) for c in readback))
print('max deviation: %.2e deg' % max(abs(a - b) for a, b in zip(target.chi, readback)))

print(f'\nMMFF energy before: {mmff_energy(peptide.mol):9.2f} kcal/mol')
print(f'MMFF energy after : {mmff_energy(trial.mol):9.2f} kcal/mol')
print('\nOne torsion change can swing the energy enormously - that is the whole')
print('reason rotamer choice matters.')

## 5. Score every rotamer of one residue

`score_residue_rotamers` places each library rotamer on a copy of the peptide and evaluates the whole-molecule MMFF energy, returning the list sorted best-first.

In [ ]:
scores = score_residue_rotamers(peptide, 1, max_chi=2)     # LYS1

for s in scores:
    chi_txt = ', '.join(f'{c:+.0f}' for c in s.rotamer.chi)
    print(f'  {s.rotamer.name}  chi=({chi_txt})   E = {s.energy:9.2f} kcal/mol')

names = [s.rotamer.name for s in scores]
vals = [s.energy for s in scores]
plt.figure(figsize=(7, 4))
plt.bar(names, vals, color=['tab:green'] + ['tab:blue'] * (len(vals) - 1))
plt.xlabel('LYS1 rotamer'); plt.ylabel('whole-molecule MMFF energy (kcal/mol)')
plt.title('rotamer energy scan for LYS1 (best in green)')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

print(f'spread between best and worst: {max(vals) - min(vals):.1f} kcal/mol')

## 6. Greedy construction, then minimization

The simplest strategy (SCWRL-like): sweep over residues, and set each to whichever rotamer minimizes the energy given where the others currently sit. A few passes let residues re-optimize against their updated neighbours. Then relax with the backbone held fixed, so only side chains move.

In [ ]:
greedy = build_low_energy_conformation(peptide, max_chi=2, verbose=True)

print('\nchosen rotamers:', greedy.assignments)
print(f'\n  embedded start : {greedy.energy_initial:9.2f} kcal/mol')
print(f'  after rotamers : {greedy.energy_constructed:9.2f} kcal/mol')
print(f'  after minimize : {greedy.energy_minimized:9.2f} kcal/mol')

stages = ['embedded\nstart', 'rotamers\nplaced', 'minimized']
evals = [greedy.energy_initial, greedy.energy_constructed, greedy.energy_minimized]
plt.figure(figsize=(6, 4))
plt.bar(stages, evals, color=['0.7', 'tab:orange', 'tab:green'])
plt.ylabel('MMFF energy (kcal/mol)'); plt.title('energy through the pipeline')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## 7. The decomposable energy matrix

Greedy is fast but shortsighted: it commits to residue *i* before knowing what residue *j* will do. To do better we need to reason about **all** combinations at once, and for that we need an energy we can precompute.

The trick is a **decomposable** energy - every term touches at most two residues:

$$ E(\text{choice}) = \sum_i E_{self}(i, r_i) \;+\; \sum_{i<j} E_{pair}(i, r_i;\, j, r_j) $$

- `E_self` - side chain *i* against the fixed backbone template, plus its own internal strain.
- `E_pair` - side chain *i* against side chain *j*.

`build_energy_matrix` computes these once with a Lennard-Jones model. After that, scoring any of the thousands of combinations is pure table lookup - no more force-field calls.

In [ ]:
matrix = build_energy_matrix(peptide, max_chi=2)

print('flexible residues:', [f'{matrix.resnames[i]}{i}' for i in matrix.resnums])
print('\nself-energies (kcal/mol):')
for i in matrix.resnums:
    row = ' '.join(f'{v:7.2f}' for v in matrix.e_self[i])
    print(f'  {matrix.resnames[i]}{i}: {row}')

print('\npair blocks stored:', sorted(matrix.e_pair.keys()))

In [ ]:
# The two phenylalanines are neighbours, so their side chains interact strongly.
i, j = matrix.resnums[-2], matrix.resnums[-1]
block = matrix.e_pair[(i, j)]
labels_i = [r.name for r in matrix.rotamers[i]]
labels_j = [r.name for r in matrix.rotamers[j]]

plt.figure(figsize=(6.5, 5.5))
plt.imshow(block, cmap='coolwarm', origin='upper')
plt.colorbar(label='pair energy (kcal/mol)')
plt.xticks(range(len(labels_j)), labels_j); plt.yticks(range(len(labels_i)), labels_i)
plt.xlabel(f'{matrix.resnames[j]}{j} rotamer'); plt.ylabel(f'{matrix.resnames[i]}{i} rotamer')
plt.title(f'E_pair block for {matrix.resnames[i]}{i} x {matrix.resnames[j]}{j}')
for a in range(block.shape[0]):
    for b in range(block.shape[1]):
        plt.text(b, a, f'{block[a, b]:.1f}', ha='center', va='center', fontsize=7)
plt.tight_layout(); plt.show()

print(f'range: {block.min():.2f} to {block.max():.2f} kcal/mol')
print('Red cells are rotamer pairs that clash. A greedy sweep that fixes one of')
print('these residues first can be trapped by exactly this coupling.')

## 8. Dead-End Elimination: provable pruning

DEE throws away rotamers that **cannot possibly** be in the global optimum. Rotamer `r` at residue `i` is dead if some alternative `t` is always at least as good, no matter what the other residues do (the Goldstein criterion):

$$ E_{self}(i,r) - E_{self}(i,t) + \sum_{j \neq i} \min_s \left[ E_{pair}(i,r;j,s) - E_{pair}(i,t;j,s) \right] > 0 $$

The `min` over `s` is the pessimistic case for `t`. If `r` still loses, it is provably dead. This is **not** a heuristic - it never discards the optimum, so it shrinks the search space for free.

In [ ]:
allowed = dead_end_elimination(matrix, verbose=False)

before = [len(matrix.rotamers[i]) for i in matrix.resnums]
after = [len(allowed[i]) for i in matrix.resnums]
labels = [f'{matrix.resnames[i]}{i}' for i in matrix.resnums]

space_before = int(np.prod(before))
space_after = int(np.prod(after))
print(f'search space before DEE: {space_before:,}')
print(f'search space after  DEE: {space_after:,}')
print(f'reduction factor       : {space_before / space_after:,.1f}x  (with no risk of losing the optimum)')

x = np.arange(len(labels))
plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, before, 0.4, label='all rotamers')
plt.bar(x + 0.2, after, 0.4, label='DEE survivors')
plt.xticks(x, labels); plt.ylabel('rotamers'); plt.legend()
plt.title('DEE prunes rotamers that cannot be optimal')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

## 9. Simulated annealing, and the provable answer

Simulated annealing proposes single-residue moves and accepts uphill ones with probability `exp(-dE/T)`, cooling `T` geometrically so it explores early and settles late.

Because this peptide is small we can also just enumerate **every** combination and get the provably optimal packing. That gives us a yardstick: we can check what SA finds, and confirm that DEE never threw the optimum away.

We run SA twice - over the full rotamer space (a genuine search) and over the DEE survivors - so you can see what the pruning bought.

In [ ]:
# SA over the FULL rotamer space - a genuine search.
choice_all, e_all = simulated_annealing(matrix, candidates=None, n_steps=4000, seed=1)
# SA restricted to the DEE survivors.
choice_sa, e_sa = simulated_annealing(matrix, candidates=allowed, n_steps=4000, seed=1)

# Exhaustive search: the provable optimum (only tractable because this peptide is tiny).
best_choice, best_e = None, np.inf
for combo in itertools.product(*(range(len(matrix.rotamers[i])) for i in matrix.resnums)):
    cand = dict(zip(matrix.resnums, combo))
    e = matrix.total_energy(cand)
    if e < best_e:
        best_e, best_choice = e, cand

def show(tag, choice, e):
    names = {f'{matrix.resnames[i]}{i}': matrix.rotamers[i][choice[i]].name
             for i in matrix.resnums}
    print(f'{tag:24s} E = {e:8.3f}   {names}')

show('SA over all rotamers', choice_all, e_all)
show('SA over DEE survivors', choice_sa, e_sa)
show(f'exhaustive ({space_before:,})', best_choice, best_e)

print(f'\nSA-over-all reached the optimum : {np.isclose(e_all, best_e)}')
print(f'optimum survived DEE pruning    : {all(best_choice[i] in allowed[i] for i in matrix.resnums)}')
print('  ^ this must ALWAYS be True - it is exactly what DEE guarantees.')

if space_after == 1:
    print(f'\nNote for this peptide: DEE left only {space_after} combination, so it solved the')
    print('problem outright and the SA-over-survivors run had nothing left to search.')
    print('That is a feature of how small this system is, not something to expect in')
    print('general - on a real protein DEE leaves many survivors and SA does the work.')

## 10. Compare the strategies end to end

`solve_rotamers` runs the whole pipeline (matrix -> selection -> place rotamers -> MMFF minimize) for a chosen method. Note the *packing* energy (the LJ score used for selection) and the final *MMFF* energy are different scales - selection and relaxation use different models on purpose.

In [ ]:
rows = [('greedy', None, greedy.energy_constructed, greedy.energy_minimized, greedy.assignments)]
results = {'greedy': greedy}
for method in ('sa', 'dee', 'dee+sa'):
    r = solve_rotamers(peptide, method=method, max_chi=2, sa_steps=4000, seed=1)
    results[method] = r
    rows.append((method, r.packing_energy, r.energy_constructed, r.energy_minimized, r.assignments))

print(f"{'method':>8} {'packing':>9} {'constructed':>12} {'minimized':>11}   assignment")
for name, pack, con, mini, assign in rows:
    pack_txt = '  -' if pack is None else f'{pack:9.2f}'
    print(f'{name:>8} {pack_txt:>9} {con:12.2f} {mini:11.2f}   {assign}')

plt.figure(figsize=(7, 4))
labels = [r[0] for r in rows]
plt.bar(labels, [r[3] for r in rows], color='tab:blue')
plt.ylabel('final minimized MMFF energy (kcal/mol)')
plt.title('side-chain packing strategies')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

best_name = min(rows, key=lambda r: r[3])[0]
print(f'lowest final MMFF energy : {best_name}')
print(f'lowest packing (LJ) energy: dee+sa at {results["dee+sa"].packing_energy:.2f}')
print('\nDo not be surprised if greedy ties or even edges out dee+sa here. The')
print('combinatorial solvers provably minimize the LJ PACKING energy, but the')
print('table ranks by MMFF energy after relaxation - two different models, so the')
print('LJ winner need not be the MMFF winner. On a 5-residue peptide the gap is a')
print('fraction of a kcal/mol, i.e. noise. The advantage of the combinatorial')
print('solvers shows up with size and crowding - which is what E4 asks you to test.')

## 11. Export for PyMOL / VMD

Write the embedded start and the best packed structure so you can look at the side chains.

In [ ]:
os.makedirs('examples/output', exist_ok=True)
best = results[best_name]
peptide.write_pdb('examples/output/nb_peptide_start.pdb')
best.peptide.write_pdb('examples/output/nb_peptide_packed.pdb')
print('wrote nb_peptide_start.pdb and nb_peptide_packed.pdb to examples/output/')

print('\nfinal chi angles of the packed structure:')
for res in best.peptide.residues:
    if res.n_chi:
        chi_txt = ', '.join(f'{c:+7.1f}' for c in best.peptide.get_all_chi(res.number))
        print(f'  {res.name}{res.number}: {chi_txt}')
print('\n(These drift off the ideal +-60/180 values because the final MMFF')
print(' minimization relaxes them - the library only supplies a starting point.)')

## Exercises

Fill in the `...` and run. Compare your results with a neighbour!

**E1. Does `max_chi` pay for itself?** Varying more chi angles gives a richer library but costs `3**max_chi` rotamers per residue. Solve with `max_chi` = 1, 2 and 3 and compare the final energy against the run time.

In [ ]:
import time
for m in (1, 2, 3):
    t0 = time.time()
    r = solve_rotamers(peptide, method='dee+sa', max_chi=m, sa_steps=4000, seed=1)
    # TODO: print m, r.energy_minimized and time.time() - t0
    ...

**E2. Is simulated annealing reliable?** SA is stochastic, so different seeds can land in different places. Run `simulated_annealing` over the **full** rotamer space (`candidates=None`) for 10 seeds and count how often it reaches the exhaustive optimum `best_e` from section 9.

In [ ]:
hits = 0
for s in range(10):
    _, e = simulated_annealing(matrix, candidates=None, n_steps=4000, seed=s)
    # TODO: increment `hits` when e is close to best_e (use np.isclose)
    ...
# TODO: print how many of the 10 seeds found the optimum

**E3. Break annealing on purpose.** SA needs enough steps to cool properly. Re-run it over the full space with `n_steps` = 20, 100, 500 and 4000, and plot the best packing energy against `n_steps`. Where does it stop improving?

In [ ]:
step_counts = (20, 100, 500, 4000)
found = []
for n in step_counts:
    # TODO: run simulated_annealing(matrix, candidates=None, n_steps=n, seed=3)
    #       and append its energy to `found`
    ...
# TODO: plot `found` against `step_counts` and draw a horizontal line at best_e

**E4. A harder peptide.** Try a sequence with bulkier, more coupled side chains, e.g. `'WRWRW'` or `'FFFFF'`. Does the gap between greedy and `dee+sa` widen? Why would you expect crowding to hurt greedy more?

In [ ]:
for seq in ('KLVFF', 'WRWRW'):
    pep = Peptide.from_sequence(seq)
    g = build_low_energy_conformation(pep, max_chi=2)
    d = solve_rotamers(pep, method='dee+sa', max_chi=2, sa_steps=4000, seed=1)
    # TODO: print seq, g.energy_minimized, d.energy_minimized and the gap
    ...

---

### Where to go next

- **Backbone-dependent library:** real rotamer preferences depend on (phi, psi). Condition the library on the backbone instead of using fixed staggered angles.
- **Rotamer probabilities:** add a `-kT ln p(rotamer)` term so statistically common rotamers are favoured, not just sterically comfortable ones.
- **Better DEE:** implement split-DEE or pair-elimination to prune harder on larger systems.
- **Electrostatics:** the packing energy here is Lennard-Jones only, so it cannot see salt bridges or hydrogen bonds. Add a Coulomb term and see whether ARG/GLU choices change.

See the closing section of `Rotamer_tutorial.md` for more.